# Causal Inference Engine — Exploratory Data Analysis

This notebook walks through:
1. Loading and inspecting the synthetic dataset
2. Confounding bias illustration
3. Propensity score estimation
4. Full causal analysis (ATE + CATE + refutation)
5. Counterfactual simulation

In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

## 1. Load Data

In [ ]:
from data_loader import load_data, validate_columns, handle_missing_values, describe_data

df = load_data('../data/sample_data.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Variable specification
TREATMENT   = 'treatment'
OUTCOME     = 'outcome'
CONFOUNDERS = ['age', 'income', 'education', 'health_score']

validate_columns(df, TREATMENT, OUTCOME, CONFOUNDERS)
df = handle_missing_values(df)

stats = describe_data(df, TREATMENT, OUTCOME, CONFOUNDERS)
print('Treatment rate  :', f"{stats['treatment_rate']:.1%}")
print('Naive ATE (biased):', f"{stats['naive_ate']:+.3f}")

## 2. Confounding Bias Illustration

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confounder imbalance
for i, feat in enumerate(['age', 'income']):
    ax = axes[i]
    df.groupby('treatment')[feat].plot.kde(ax=ax, legend=True)
    ax.set_title(f'Distribution of {feat} by treatment status')
    ax.set_xlabel(feat)

plt.tight_layout()
plt.show()
print('Notice: treated units tend to be older and have higher income → confounding!')

## 3. Build Causal Model

In [ ]:
from causal_model import build_causal_model

ci_model = build_causal_model(df, TREATMENT, OUTCOME, CONFOUNDERS)
print('Identified estimand:\n')
print(ci_model.identified_estimand)

In [ ]:
from visualization import plot_dag

fig = plot_dag(TREATMENT, OUTCOME, CONFOUNDERS, dag=ci_model.dag)
plt.show()

## 4. ATE Estimation

In [ ]:
from estimator import estimate_ate_linear, estimate_ate_propensity, summarise_estimates

# OLS
ols_result = estimate_ate_linear(df, TREATMENT, OUTCOME, CONFOUNDERS)
print(f"OLS ATE  = {ols_result['ate']:+.4f}  (95 CI: [{ols_result['ci_lower']:+.3f}, {ols_result['ci_upper']:+.3f}])")
print(f"p-value  = {ols_result['p_value']:.4f}")

In [ ]:
# Propensity Score Matching
psm_result = estimate_ate_propensity(ci_model)
print(f"PSM ATE ({psm_result['method']}) = {psm_result['ate']:+.4f}")

## 5. Heterogeneous Effects (CATE)

In [ ]:
from estimator import estimate_heterogeneous_effects

cate_result = estimate_heterogeneous_effects(df, TREATMENT, OUTCOME, CONFOUNDERS)
print(f"Mean CATE = {cate_result['cate_mean']:+.4f}  (std={cate_result['cate_std']:.4f})")

In [ ]:
from visualization import plot_treatment_effect_distribution, plot_cate_by_feature

fig = plot_treatment_effect_distribution(cate_result['cate_values'], ate=cate_result['cate_mean'])
plt.show()

fig2 = plot_cate_by_feature(cate_result['X'], cate_result['cate_values'])
plt.show()

## 6. Refutation Tests

In [ ]:
from refutation import run_random_common_cause, run_placebo_test, run_data_subset_refuter, interpret_refutation

ols_est = ci_model.estimate(method_name='backdoor.linear_regression')

rcc     = run_random_common_cause(ci_model, ols_est, n_simulations=10)
placebo = run_placebo_test(ci_model, ols_est, n_simulations=10)
subset  = run_data_subset_refuter(ci_model, ols_est, n_simulations=10)

print(interpret_refutation([rcc, placebo, subset]))

## 7. Counterfactual Simulation

In [ ]:
from simulator import simulate_counterfactuals
from visualization import plot_counterfactual_distributions, plot_policy_comparison

sim = simulate_counterfactuals(df, TREATMENT, OUTCOME, CONFOUNDERS)

print(f"E[Y(0)] = {sim['mean_y0']:.4f}")
print(f"E[Y(1)] = {sim['mean_y1']:.4f}")
print(f"E[ITE]  = {sim['mean_ite']:+.4f}")

fig = plot_counterfactual_distributions(sim['y0_hat'], sim['y1_hat'], df[OUTCOME].values)
plt.show()

fig2 = plot_policy_comparison(sim['policy_df'])
plt.show()

print('\nPolicy comparison:')
sim['policy_df'].round(3)

## 8. Compare with Oracle (Ground Truth)

Since this is synthetic data, we can check our estimate against the known truth.

In [ ]:
oracle = pd.read_csv('../data/sample_data_oracle.csv')
true_ate = oracle['_true_ite'].mean()

print('═' * 45)
print(f'  True ATE  (oracle)    : {true_ate:+.4f}')
print(f'  Naive ATE (biased)    : {stats["naive_ate"]:+.4f}')
print(f'  OLS ATE   (adjusted)  : {ols_result["ate"]:+.4f}')
print(f'  PSM ATE               : {psm_result["ate"]:+.4f}')
print(f'  Mean CATE (DML)       : {cate_result["cate_mean"]:+.4f}')
print('─' * 45)
print(f'  OLS error vs truth    : {abs(ols_result["ate"] - true_ate):.4f}')
print(f'  Naive error vs truth  : {abs(stats["naive_ate"] - true_ate):.4f}  ← confounding bias')
print('═' * 45)